In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os
import warnings
import sys

"""
Computes classification metrics (accuracy, recall, confusion matrices) comparing 'eye' and 'after lift' human predictions against ground truth.
"""

print("--- Script Started ---")

# --- 1. Path Definition and Setup ---
try:
    # Set the base path relative to the current script location
    BASE_PATH = Path.cwd().parent 
    DATA_DIR = BASE_PATH / 'data'
    OUTPUT_DIR = DATA_DIR / 'all' / 'questionnaire'
    
    # Create output directory if it doesn't exist
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    COMBINED_CSV_PATH = OUTPUT_DIR / 'combined_dataset.csv'
    METRICS_CSV_PATH = OUTPUT_DIR / 'accuracy_report.csv'

    print(f"Base Directory (..): {BASE_PATH.resolve()}")
    print(f"Data Directory (input): {DATA_DIR.resolve()}")
    print(f"Report Directory (output): {OUTPUT_DIR.resolve()}")

except Exception as e:
    print(f"Fatal error setting up paths: {e}")
    sys.exit(1)


# --- 2. File Search, Extraction, and Data Merging ---
print("\n--- 2. Data Search and Extraction ---")

# Define pattern to find specific label files recursively
search_pattern = 'S*/test_1*/opaq/labels.csv'
print(f"Starting file search with pattern: {DATA_DIR / search_pattern}")

file_paths = list(DATA_DIR.glob(search_pattern))

if not file_paths:
    print("\nWARNING: No 'labels.csv' files found.")
    print("Please check that the folder structure is correct.")
    print("The script will now stop.")
    sys.exit(0)
else:
    print(f"Found {len(file_paths)} 'labels.csv' files. Starting processing...")

required_cols = ['label_real', 'label_eye', 'label_after']
all_dataframes = []
files_with_errors = 0

# Loop through all found files to read and tag them
for f_path in file_paths:
    try:
        df = pd.read_csv(f_path, usecols=required_cols)
        
        # Extract metadata (subject and test) from the file path
        relative_path_parts = f_path.relative_to(DATA_DIR).parts
        df['source_subject'] = relative_path_parts[0]
        df['source_test'] = relative_path_parts[1]
        
        all_dataframes.append(df)
        
    except Exception as e:
        print(f"ERROR in {f_path}: {e}.")
        files_with_errors += 1

print(f"File processing complete. Files read: {len(all_dataframes)}, Files with errors: {files_with_errors}")

if all_dataframes:
    # Concatenate all individual dataframes into one master dataframe
    master_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"\nCombined DataFrame created successfully. Total rows: {len(master_df)}")
    
    try:
        master_df.to_csv(COMBINED_CSV_PATH, index=False)
        print(f"Full DataFrame saved to: {COMBINED_CSV_PATH}")
    except Exception as e:
        print(f"Error saving combined DataFrame: {e}")
        
    print("\nFirst 5 rows of the combined DataFrame:")
    print(master_df.head())
    
else:
    print("\nNo data loaded. Script cannot continue.")
    sys.exit(0)


# --- 3. Calculation of Accuracy Metrics ---
print("\n--- 3. Accuracy Metrics Calculation ---")

if 'master_df' in locals() and not master_df.empty:
    
    # Define Ground Truth and Predictions
    y_true = master_df['label_real']
    y_pred_eye = master_df['label_eye']
    y_pred_after = master_df['label_after']
    
    # Define classes for classification report
    labels = [1, 2, 3]
    target_names = ['1_light', '2_medium', '3_heavy']

    print("\n--- 3a. Overall Accuracy ---")
    
    acc_eye = accuracy_score(y_true, y_pred_eye)
    acc_after = accuracy_score(y_true, y_pred_after)
    
    print(f"Accuracy 'label_eye' (Before): {acc_eye:.4f}")
    print(f"Accuracy 'label_after' (After): {acc_after:.4f}")

    # Generate detailed classification reports as dictionaries
    report_eye_dict = classification_report(
        y_true, y_pred_eye, labels=labels, target_names=target_names, 
        output_dict=True, zero_division=0
    )
    report_after_dict = classification_report(
        y_true, y_pred_after, labels=labels, target_names=target_names, 
        output_dict=True, zero_division=0
    )

    # --- Per-Label Accuracy (Recall) ---
    print("\n--- 3b. Accuracy per Single Label (Recall) ---")
    print("(How many cases of a specific class were correctly identified?)")
    
    print("\n'label_eye' (Before):")
    for label_name in target_names:
        recall_val = report_eye_dict[label_name]['recall']
        print(f"  - Accuracy (Recall) for '{label_name}': {recall_val:.4f}")

    print("\n'label_after' (After):")
    for label_name in target_names:
        recall_val = report_after_dict[label_name]['recall']
        print(f"  - Accuracy (Recall) for '{label_name}': {recall_val:.4f}")


    # --- Confusion Matrices ---
    print("\n--- 3c. Numerical Confusion Matrices ---")
    print("(Rows represent True classes, Columns represent Predicted classes)")

    # Compute matrices
    cm_eye = confusion_matrix(y_true, y_pred_eye, labels=labels)
    cm_after = confusion_matrix(y_true, y_pred_after, labels=labels)
    
    # Format with Pandas for better readability in output
    df_cm_eye = pd.DataFrame(cm_eye, 
                             index=[f"True_{l}" for l in target_names], 
                             columns=[f"Pred_{l}" for l in target_names])
    
    df_cm_after = pd.DataFrame(cm_after, 
                               index=[f"True_{l}" for l in target_names], 
                               columns=[f"Pred_{l}" for l in target_names])

    print("\nConfusion Matrix 'label_eye' (Before):")
    print(df_cm_eye)
    
    print("\nConfusion Matrix 'label_after' (After):")
    print(df_cm_after)


    # --- 4. Saving Metrics ---
    print("\n--- 4. Saving Full Metrics Report ---")
    
    try:
        # Convert report dictionaries to DataFrames
        df_report_eye = pd.DataFrame(report_eye_dict).transpose()
        df_report_after = pd.DataFrame(report_after_dict).transpose()
        
        # Merge the two reports for side-by-side comparison
        df_metrics = pd.concat(
            [df_report_eye, df_report_after], 
            keys=['prediction_EYE', 'prediction_AFTER'], 
            axis=1
        )
              
        # Save to CSV
        df_metrics.to_csv(METRICS_CSV_PATH)
        
        print(f"Combined metrics report saved to: {METRICS_CSV_PATH}")
        print("\nContent of the metrics report (same as CSV file):")
        print(df_metrics)
        
    except Exception as e:
        print(f"Error saving metrics report: {e}")

else:
    print("ERROR: Variable 'master_df' is undefined or empty.")

print("\n--- Script Completed ---")

--- Inizio Script ---
Cartella Base (..): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv
Cartella Dati (input): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data
Cartella Report (output): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\questionnaire

--- 2. Ricerca ed Estrazione Dati ---
Inizio ricerca file con pattern: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S*\test_1*\opaq\labels.csv
Trovati 12 file 'labels.csv'. Inizio processamento...
Processamento file completato. File letti: 12, File con errori: 0

DataFrame combinato creato con successo. Totale righe: 108
DataFrame completo salvato in: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\questionnaire\combined_dataset.csv

Prime 5 righe del DataFrame combinato:
   label_real  label_eye  label_after source_subject source_test
0           2          2            2            S03      test_1
1           3          1            3            S03      test_1
2           1          3            1            S03      test